# Infini-gram Corpus Occurrence vs. Citation Count

Reproduces the LLM corpus occurrence analysis (Section 3.2 and Appendix "More Results on Citation As a Proxy").
For each result file produced by `query_infini_gram.py` this notebook computes

1. the Pearson / Spearman correlation between citation count and n-gram count (raw and log2 scale),
2. a log2-scale scatter plot with a linear fit,
3. n-gram count statistics per citation bucket (count, mean, std, quantiles, coefficient of variation).

Result files in the paper:

| File | Paper table |
|---|---|
| `infini_gram_olmo2_capitalize.csv` | OLMo2-32B, capitalized title (main result, Figure 2 right) |
| `infini_gram_redpajama_capitalize.csv` | RedPajama, capitalized title |
| `infini_gram_olmo2_capitalize_resample.csv` | OLMo2-32B, independent resample (robustness) |
| `infini_gram_olmo2_camel.csv`, `infini_gram_redpajama_camel.csv` | CamelCase title (query-format sensitivity) |

Bucket statistics are computed on the raw counts (zeros kept as zeros).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import LogLocator
from scipy.stats import pearsonr, spearmanr
from tabulate import tabulate

sys.path.insert(0, str(Path.cwd().parent))  # run this notebook from its own folder
import config

IN_DIR = config.INFINI_GRAM_DIR
result_files = sorted(IN_DIR.glob("infini_gram_*.csv"))
result_files = [f for f in result_files if not f.name.endswith(".partial.csv")]
print("Available result files:", [f.name for f in result_files])

In [ ]:
def correlations(df: pd.DataFrame) -> dict[str, float]:
    """Pearson and Spearman correlations on the raw and the log2 scale (zeros replaced by 0.1)."""
    x_log = np.log2(df["citationCount"].replace(0, 0.1))
    y_log = np.log2(df["count"].replace(0, 0.1))
    return {
        "pearson": pearsonr(df["citationCount"], df["count"])[0],
        "spearman": spearmanr(df["citationCount"], df["count"])[0],
        "pearson_log2": pearsonr(x_log, y_log)[0],
        "spearman_log2": spearmanr(x_log, y_log)[0],
    }


def log2_scatter(df: pd.DataFrame, title: str, out_path: Path) -> None:
    """Scatter plot of citation count vs. n-gram count on log2 axes with a linear fit."""
    x = df["citationCount"].replace(0, 0.1)
    y = df["count"].replace(0, 0.1)
    slope, intercept = np.polyfit(np.log2(x), np.log2(y), 1)
    x_fit = np.linspace(np.log2(x).min(), np.log2(x).max(), 500)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(x, y, alpha=0.3, color="blue")
    ax.plot(2 ** x_fit, 2 ** (slope * x_fit + intercept), color="red", linewidth=2, label="Linear fit (log2 space)")
    ax.set_xscale("log", base=2)
    ax.set_yscale("log", base=2)
    ax.xaxis.set_major_locator(LogLocator(base=2, numticks=10))
    ax.yaxis.set_major_locator(LogLocator(base=2, numticks=10))
    ax.set_xlabel("Citation (Log2 Scale)")
    ax.set_ylabel("N-gram Count (Log2 Scale)")
    ax.set_title(title)
    ax.grid(True, which="both", ls="--", linewidth=0.5)
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.show()


def bucket_stats(df: pd.DataFrame) -> pd.DataFrame:
    """n-gram count statistics per citation bin, ordered from the lowest to the highest bin."""
    table = df.groupby("citation_bin").agg(
        count=("count", "count"),
        mean_count=("count", "mean"),
        std_count=("count", "std"),
        min_count=("count", "min"),
        q25_count=("count", lambda x: x.quantile(0.25)),
        q50_count=("count", "median"),
        q75_count=("count", lambda x: x.quantile(0.75)),
        max_count=("count", "max"),
    ).reindex(config.CITATION_BIN_LABELS).reset_index()
    table["cv_count"] = table["std_count"] / table["mean_count"]
    return table

## Analyse every available result file

In [ ]:
all_correlations = {}
for path in result_files:
    name = path.stem.replace("infini_gram_", "")
    df = pd.read_csv(path).dropna(subset=["count"])
    print(f"\n=== {name}: {len(df)} papers ===")

    all_correlations[name] = correlations(df)
    log2_scatter(df, f"Citation vs N-gram Count ({name})", IN_DIR / f"scatter_{name}.pdf")

    table = bucket_stats(df)
    table.to_csv(IN_DIR / f"citation_bucket_stats_{name}.csv", index=False)
    print(tabulate(table, headers="keys", tablefmt="github", showindex=False, floatfmt=".2f"))

pd.DataFrame(all_correlations).T.round(4)

## Robustness: original sample vs. independent resample

Paper-level Spearman correlation between citation count and n-gram count (OLMo2-32B, capitalized
title) for the original proxy set and for the resample (reported as 0.39 in the paper), next to
the per-bucket means of both samples.

In [ ]:
pairs = {"original": "olmo2_capitalize", "resample": "olmo2_capitalize_resample"}
if all(name in all_correlations for name in pairs.values()):
    for label, name in pairs.items():
        print(f"{label:9s} Spearman(citation, n-gram count) = {all_correlations[name]['spearman']:.2f}")
    means = pd.DataFrame({
        label: pd.read_csv(IN_DIR / f"citation_bucket_stats_{name}.csv")["mean_count"].values
        for label, name in pairs.items()
    }, index=config.CITATION_BIN_LABELS)
    display(means.round(2))
else:
    print("Run query_infini_gram.py --corpus olmo2 --query-format capitalize with and without --resample first.")